# LIBRARY

In [33]:
import json
import re

REQUIRED_FIELDS = ["time", "src", "dst", "protocol", "length", "info"]
VALID_PROTOCOLS = ["ZigBee", "ZigBee HA"]

def validate_packet(line):
    """
    Attempt to validate a single packet line.
    Returns tuple (is_valid, error_message, packet_dict_or_None)
    """

    original_line = line.strip()

    # Skip empty lines
    if not original_line:
        return False, "Empty line", None

    # Try JSON load
    try:
        # Remove trailing commas if present:
        cleaned = original_line.rstrip(",")
        pkt = json.loads(cleaned)
    except Exception as e:
        return False, f"JSON parse error: {e}", None

    # Check required fields
    for field in REQUIRED_FIELDS:
        if field not in pkt:
            return False, f"Missing required field: {field}", pkt

    # Detect spelling mistakes
    wrong_fields = [f for f in pkt.keys() if f not in REQUIRED_FIELDS]
    if wrong_fields:
        return False, f"Unexpected field(s): {wrong_fields}", pkt

    # # Check duplicate keys (regex)
    # duplicate_keys = re.findall(r'"(\w+)":.*"(\w+)":', original_line)
    # if duplicate_keys:
    #     return False, "Duplicate key in packet", pkt

    # Time must be a valid float
    try:
        float(pkt["time"])
    except:
        return False, "Time not numeric", pkt

    # ---- Check: length is numeric ----
    try:
        int(pkt["length"])
    except:
        return False, "Length field is not numeric", pkt

    # ---- Check: protocol must be valid ----
    if pkt["protocol"] not in VALID_PROTOCOLS:
        return False, f"Invalid protocol type: {pkt['protocol']}", pkt
    return True, None, pkt


def analyze_rnn_output(filepath):
    """
    Reads raw RNN-generated text and analyzes packet validity.
    """

    with open(filepath, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # Split by new lines (each line is a JSON packet or garbage)
    lines = raw_text.split("\n")

    valid_packets = []
    corrupt_packets = []

    last_time = -1
    timestamp_errors = 0

    for i, line in enumerate(lines):
        is_valid, error, pkt = validate_packet(line)

        if is_valid:
            # Check ordering (optional)
            t = float(pkt["time"])
            if t < last_time:
                timestamp_errors += 1
            last_time = t
            valid_packets.append(pkt)
        else:
            corrupt_packets.append((i, line, error))

    total = len(lines)
    corrupt_ratio = (len(corrupt_packets) / max(1, total))*100
    decodability = 100 - corrupt_ratio

    print("\n===== RNN PACKET QUALITY REPORT =====")
    print(f"Total lines processed: {total}")
    print(f"Valid packets: {len(valid_packets)}")
    print(f"Corrupt packets: {len(corrupt_packets)}")
    print(f"% Corrupt: {corrupt_ratio:.2f}%")
    print(f"Timestamp ordering issues: {timestamp_errors}")
    print("=====================================\n")

    if corrupt_packets:
        print("---- CORRUPT PACKETS ----")
        for idx, line, reason in corrupt_packets[:20]:  # show only first 20
            print(f"[Line {idx}] Error: {reason}")
            print(f"    {line}\n")
    else:
        print("No corrupt packets detected.\n")

    return valid_packets, corrupt_packets, decodability


def check_protocol_compliance(valid_packets):
    """
    Checks whether each packet uses the correct protocol for its message type.
    Rules:
        - If 'Link Status' -> protocol must be 'ZigBee'
        - Otherwise       -> protocol must be 'ZigBee HA'
    """

    incorrect = []

    for i, pkt in enumerate(valid_packets):
        info = pkt["info"]
        protocol = pkt["protocol"]

        if "Link Status" in info:
            expected = "ZigBee"
        else:
            expected = "ZigBee HA"

        if protocol != expected:
            incorrect.append((i, pkt, expected))

    total = len(valid_packets)
    error_count = len(incorrect)
    compliance = 100 * (1 - error_count / total) if total > 0 else 0

    print("\n===== PROTOCOL COMPLIANCE REPORT =====")
    print(f"Total packets: {total}")
    print(f"Incorrect protocol packets: {error_count}")
    print(f"Protocol Compliance: {compliance:.2f}%")
    print("======================================\n")

    if incorrect:
        print("---- Incorrect Protocol Packets ----")
        for idx, pkt, expected in incorrect[:20]:
            print(f"[Packet {idx}] Expected: {expected}, Found: {pkt['protocol']}")
            print(f"   Info: {pkt['info']}")
            print(f"   Full Packet: {pkt}\n")

    return compliance, incorrect

def check_address_compliance(valid_packets):
    """
    Checks address correctness:
        - Link Status -> dst must be 0xfffc (broadcast)
        - src != dst
    """

    incorrect = []

    for i, pkt in enumerate(valid_packets):
        src = pkt["src"]
        dst = pkt["dst"]
        info = pkt["info"]

        # Rule 1: Link Status must be broadcast
        if "Link Status" in info:
            if dst != "0xfffc":
                incorrect.append((i, pkt, "dst must be 0xfffc for Link Status"))

        # Rule 2: src and dst cannot be equal
        if src == dst:
            incorrect.append((i, pkt, "src and dst cannot be identical"))

    total = len(valid_packets)
    error_count = len(incorrect)
    compliance = 100 * (1 - error_count / total) if total > 0 else 0

    print("\n===== ADDRESS COMPLIANCE REPORT =====")
    print(f"Total packets: {total}")
    print(f"Packets violating address rules: {error_count}")
    print(f"Address Compliance: {compliance:.2f}%")
    print("=====================================\n")

    if incorrect:
        print("---- Incorrect Address Packets ----")
        for idx, pkt, reason in incorrect[:20]:
            print(f"[Packet {idx}] Error: {reason}")
            print(f"   Full Packet: {pkt}\n")

    return compliance, incorrect


    """
    Checks if ZCL sequence numbers increase monotonically after sorting by time.
    
    Expected format:
        "info": "ZCL: Read Attributes Response, Seq: 24"
    """

    # First: sort packets by time
    sorted_packets = sorted(valid_packets, key=lambda p: float(p["time"]))

    seq_packets = []
    seq_values = []

    # Extract sequence numbers
    for pkt in sorted_packets:
        info = pkt["info"]

        # Only handle ZCL response messages
        if "ZCL: Read Attributes Response" in info:
            match = re.search(r"Seq:\s*(\d+)", info)
            if match:
                seq = int(match.group(1))
                seq_packets.append(pkt)
                seq_values.append(seq)

    # Now check ordering
    incorrect = []
    for i in range(1, len(seq_values)):
        if seq_values[i] < seq_values[i - 1]:
            incorrect.append((i, seq_packets[i], seq_values[i - 1], seq_values[i]))

    total = len(seq_values)
    error_count = len(incorrect)
    compliance = 100 * (1 - error_count / total) if total > 0 else 0

    print("\n===== ZCL SEQUENCE ORDER COMPLIANCE =====")
    print(f"Total ZCL Response Packets: {total}")
    print(f"Out-of-order sequence packets: {error_count}")
    print(f"Sequence Compliance: {compliance:.2f}%")
    print("=========================================\n")

    if incorrect:
        print("---- Out-of-order SEQ Packets ----")
        for idx, pkt, prev_seq, curr_seq in incorrect[:20]:
            print(f"[Packet sorted index {idx}] Expected seq > {prev_seq}, but got {curr_seq}")
            print(f"   Time: {pkt['time']}")
            print(f"   Info: {pkt['info']}")
            print(f"   Full Packet: {pkt}\n")

    return compliance, incorrect


    """
    Finds duplicate packets based on (src, dst, protocol, info).
    Returns:
        - repetition_percentage (float)
        - repeated_entries (list of (packet, count))
    """

    from collections import Counter

    # Create hashable packet signature
    signatures = [
        (pkt["src"], pkt["dst"], pkt["protocol"], pkt["info"])
        for pkt in packets
    ]

    counter = Counter(signatures)

    # Find repeated entries (count > 1)
    repeated = [(sig, count) for sig, count in counter.items() if count > 1]

    total = len(packets)
    repeated_packet_count = sum(count - 1 for _, count in repeated)  # excludes first occurrence

    repetition_percentage = 100 * repeated_packet_count / max(1, total)

    print("\n===== PACKET REPETITION ANALYSIS =====")
    print(f"Total packets: {total}")
    print(f"Repeated packet patterns: {len(repeated)}")
    print(f"Total repeated occurrences (beyond first): {repeated_packet_count}")
    print(f"Repetition Percentage: {repetition_percentage:.2f}%")
    print("======================================\n")

    if repeated:
        print("---- Repeated Packet Patterns ----")
        for (sig, count) in repeated[:20]:  # print first 20 repeats
            src, dst, protocol, info = sig
            print(f"Repeated {count} times:")
            print(f"  src={src}, dst={dst}, protocol={protocol}")
            print(f"  info={info}\n")

    return repetition_percentage, repeated

In [34]:
import re

def check_seq_ordering(valid_packets, max_step=50):
    """
    GAN-hardened ZCL sequence checker.

    Rules:
      1) Each ZCL packet must contain a numeric Seq in [0,255].
      2) Sequence must progress modulo 256 with small forward steps:
           -> allowed: 250 -> 5  (wrap-around)
           -> NOT allowed: large backward jumps (e.g., 250 -> 200)
      3) For consecutively adjacent packets with the SAME message type (same info prefix),
         the sequence number must not repeat (no immediate duplicate).
         However, the same sequence value may reappear later (non-consecutive).
    """

    # Sort by timestamp (preserves original order for reporting)
    packets_sorted = sorted(valid_packets, key=lambda p: float(p["time"]))

    seq_packets = []   # packets that contain a valid (or parseable) seq
    seq_values = []
    info_types = []
    seq_indices = []   # original index in the sorted list (for error reporting)

    incorrect = []     # list of error events: (orig_idx, pkt, prev, curr, diff, reason)

    # --- Extraction + basic validation ---
    for orig_idx, pkt in enumerate(packets_sorted):
        info = pkt.get("info", "")

        # consider ZCL packets only
        if "ZCL" in info:
            # Try to find Seq:
            m = re.search(r"Seq:\s*([+-]?\d+)", info)
            if not m:
                # Missing Seq field -> mark error and skip adding to seq lists
                incorrect.append((orig_idx, pkt, None, None, None, "Missing Seq field for ZCL packet"))
                continue

            raw_seq = m.group(1)
            # numeric?
            try:
                seq = int(raw_seq)
            except:
                incorrect.append((orig_idx, pkt, None, None, None, f"Non-numeric Seq value: {raw_seq}"))
                continue

            # range check
            if seq < 0 or seq > 255:
                incorrect.append((orig_idx, pkt, None, seq, None, f"Seq out of valid range: {seq}"))

            # store for further ordering checks
            seq_packets.append(pkt)
            seq_values.append(seq)
            # base info = everything before "Seq:"
            info_types.append(info.split("Seq:")[0].strip())
            seq_indices.append(orig_idx)

    # If no ZCL sequences found, return clean report
    total = len(seq_values)
    if total == 0:
        print("\n===== ZCL SEQUENCE ORDER REPORT (GAN) =====")
        print("Total ZCL packets: 0")
        print("No ZCL Seq values found; nothing to check.")
        print("===========================================\n")
        return 100.0, incorrect

    # --- Ordering checks ---
    for i in range(1, total):
        prev = seq_values[i - 1]
        curr = seq_values[i]
        orig_idx = seq_indices[i]
        pkt = seq_packets[i]

        # modular difference (0..255)
        diff = (curr - prev) % 256

        # Large backward jump detection: diff should be in [1, max_step]
        # Allow diff==0 only if info types differ (i.e., not consecutive same-info duplicate).
        if diff == 0:
            # same seq value consecutive

            """
            Packet 1: ZCL: Read Attributes Response, Seq: 120
            Packet 2: ZCL: Read Attributes Response, Seq: 120  ← ERROR

            """
            if info_types[i] == info_types[i - 1]:
                incorrect.append((orig_idx, pkt, prev, curr, diff,
                                  "Repeated Seq for SAME info type (consecutive duplicate)"))
            else:
                # same seq but different info type — allow (no error)
                pass
        else:
            # if diff > max_step then it's an unrealistic jump (backwards too far)
            if diff > max_step:
                incorrect.append((orig_idx, pkt, prev, curr, diff,
                                  f"Unrealistic sequence jump (diff={diff} > max_step={max_step})"))

    # --- Compute error statistics properly: unique bad packet indices ---
    error_events = len(incorrect)
    unique_error_indices = set([ev[0] for ev in incorrect if ev[0] is not None])
    error_packet_count = len(unique_error_indices)

    compliance = 100 * (1 - error_packet_count / max(1, total))

    # --- Reporting ---
    print("\n===== ZCL SEQUENCE ORDER REPORT (GAN) =====")
    print(f"Total ZCL packets (with Seq): {total}")
    print(f"Total error events: {error_events}")
    print(f"Unique packets with errors: {error_packet_count}")
    print(f"Sequence Compliance: {compliance:.2f}%")
    print("===========================================\n")

    if incorrect:
        print("---- First incorrect SEQ EVENTS (up to 20) ----")
        for orig_idx, pkt, prev, curr, diff, reason in incorrect[:20]:
            print(f"[Sorted idx {orig_idx}] Reason: {reason}")
            if prev is not None and curr is not None:
                print(f"   Prev={prev}, Curr={curr}, Jump={diff}")
            print(f"   Time: {pkt.get('time')}")
            print(f"   Info: {pkt.get('info')}\n")

    return compliance, incorrect


In [35]:
from collections import Counter

def check_repetition_rate(valid_packets, top_n=10):
    """
    Computes repetition only for NON-Link-Status ZigBee packets.
    Groups packets by (src, dst, protocol, length, info).
    """

    keys = []

    for pkt in valid_packets:
        info = pkt["info"]

        # Skip Link Status packets
        if "Link Status" in info:
            continue

        key = (pkt["src"], pkt["dst"], pkt["protocol"], pkt["length"], pkt["info"])
        keys.append(key)

    counter = Counter(keys)
    total_packets = len(keys)  # Only non-Link-Status packets

    repeated_packets = sum(count - 1 for count in counter.values() if count > 1)

    repetition_rate = (repeated_packets / max(1, total_packets)) * 100

    print("\n===== PACKET REPETITION REPORT (NO LINK-STATUS) =====")
    print(f"Total NON-Link-Status packets: {total_packets}")
    print(f"Repeated packets: {repeated_packets}")
    print(f"Repetition rate: {repetition_rate:.2f}%")
    print("======================================================\n")

    print(f"---- TOP {top_n} MOST REPEATED PATTERNS ----")
    for key, count in counter.most_common(top_n):
        if count > 1:
            src, dst, protocol, length, info = key
            print(f"{count}× | {src} → {dst} | {protocol} | len={length} | {info}")
        else:
            break

    return repetition_rate, counter, repeated_packets

In [36]:
def compute_exact_match_rate(valid_packets, sample_packets):
    """
    Computes Exact Match Rate between generated packets and real sample packets.
    Comparison rules:
        - time: only integer part must match
        - src, dst, protocol, length, info: must match exactly
    """

    def normalize(pkt):
        """Convert packet to a comparison key with integer time."""
        return (
            int(float(pkt["time"])),   # integer time only
            pkt["src"],
            pkt["dst"],
            pkt["protocol"],
            pkt["length"],
            pkt["info"]
        )

    # Convert real sample packets into a set for fast lookup
    sample_set = {normalize(pkt) for pkt in sample_packets}

    exact_matches = []
    total_gen = len(valid_packets)

    for pkt in valid_packets:
        key = normalize(pkt)
        if key in sample_set:
            exact_matches.append(pkt)

    exact_count = len(exact_matches)
    exact_match_rate = (exact_count / max(1, total_gen)) * 100

    print("\n===== EXACT MATCH RATE REPORT =====")
    print(f"Total generated packets: {total_gen}")
    print(f"Exact matches with real data: {exact_count}")
    print(f"Exact Match Rate: {exact_match_rate:.2f}%")
    print("===================================\n")

    if exact_matches:
        print("---- Example Exact-Matched Packets (first 10) ----")
        for pkt in exact_matches[:10]:
            print(pkt)
            print()

    return exact_match_rate, exact_matches

# RESULTS

## EXPERIMENT 1

In [37]:
REQUIRED_FIELDS = ["time", "src", "dst", "protocol", "length", "info"]
VALID_PROTOCOLS = ["ZigBee", "ZigBee HA"]


VALID_SOURCES = ["0x1de6"] # only for experiment #1
VALID_DESTINATIONS = ["0xd7a7", "0xfffc"] # only for experiment #1

In [38]:
import json
import os

data = []
with open('../Datasets/Experiment_1_one_way_communication_10_minute_input_sample.json', 'r') as file:
    for line in file:
        data.append(json.loads(line))

#print(data)

""" Change Broadcast with the address """
for item in data:
    if item['Destination'] == 'Broadcast':
        item['Destination'] = '0xfffc'

""" Create sample prompt from json file """
Sample_Packets= []

for item in data:
    sample_packets = {
        'time': str(item['Time']),
        'src': item['Source'],
        'dst': item['Destination'],
        'protocol': item['Protocol'],
        'length': str(item['Length']),
        'info': item['Info']
    }
    Sample_Packets.append(sample_packets)


print(Sample_Packets)

[{'time': '0.0', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link Status'}, {'time': '8.041278', 'src': '0x1de6', 'dst': '0xd7a7', 'protocol': 'ZigBee HA', 'length': '69', 'info': 'ZCL: Read Attributes Response, Seq: 216'}, {'time': '8.073592', 'src': '0x1de6', 'dst': '0xd7a7', 'protocol': 'ZigBee HA', 'length': '61', 'info': 'ZCL: Read Attributes Response, Seq: 217'}, {'time': '14.242098', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link Status'}, {'time': '20.786429', 'src': '0x1de6', 'dst': '0xd7a7', 'protocol': 'ZigBee HA', 'length': '69', 'info': 'ZCL: Read Attributes Response, Seq: 239'}, {'time': '20.812187', 'src': '0x1de6', 'dst': '0xd7a7', 'protocol': 'ZigBee HA', 'length': '61', 'info': 'ZCL: Read Attributes Response, Seq: 240'}, {'time': '28.563272', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link Status'}, {'time': '33.56803', 'src': '0x1de6', 'dst': '0xd7a7', 'p

In [39]:
def sort_by_time(packets):
    """Return packets sorted by numeric time value."""
    return sorted(packets, key=lambda p: float(p["time"]))


In [40]:
print("\n===== SUMMARY REPORT =====")

Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []

N = list(range(1,11)) # number of trial

for n in N:

    filepath = f"../Generated_Traffic/TXT_files/RNN_Exp1_Trial_{n}_raw_generated_10_minutes.txt"
    valid_packets, corrupt_packets, decodability = analyze_rnn_output(filepath)

    # ---- SORT VALID PACKETS BY TIME ----
    valid_packets = sort_by_time(valid_packets)

    # ---- METRIC COMPUTATION ----
    protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)
    address_compliance, address_errors = check_address_compliance(valid_packets)
    seq_compliance, seq_errors = check_seq_ordering(valid_packets)
    repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)
    emr, matches = compute_exact_match_rate(Sample_Packets, valid_packets)


    Decodability.append(decodability)
    Protocol_Compliance_Rate.append(protocol_compliance)
    Address_Compliance_Rate.append(address_compliance)
    Seq_Compliance_Rate.append(seq_compliance)
    Repetition_Rate.append(repetition_rate)
    Exact_Match_Rate.append(emr)
    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%, EMR: {emr:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average Exact Match Rate (EMR): {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")


===== SUMMARY REPORT =====

===== RNN PACKET QUALITY REPORT =====
Total lines processed: 137
Valid packets: 123
Corrupt packets: 14
% Corrupt: 10.22%
Timestamp ordering issues: 61

---- CORRUPT PACKETS ----
[Line 0] Error: Empty line
    

[Line 1] Error: JSON parse error: Extra data: line 1 column 12 (char 11)
    "ZigBee HA", "length": "61", "info": "ZCL: Read Httributes Response, Seq: 20"}

[Line 21] Error: Missing required field: src
    {"time": "48.910431", "rsc": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "61", "info": "ZCL: Read Attributes Response, Seq: 186"}

[Line 22] Error: Missing required field: protocol
    {"time": "48.692613", "src": "0x1de6", "dst": "0xd7a7", "prototocol": "ZigBee HA", "length": "61", "info": "ZCL: Read Attributes Response, Seq: 256"}

[Line 26] Error: Time not numeric
    {"time": "308.R779", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read Attributes Response, Seq: 43"}

[Line 58] Erro

## 30 MINUTES TRAFFIC

In [41]:
print("\n===== SUMMARY REPORT =====")

Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []

N = list(range(1,2)) # number of trial

for n in N:

    filepath = f"../Generated_Traffic/TXT_files/RNN_Exp1_Trial_raw_generated_30_minutes.txt"
    valid_packets, corrupt_packets, decodability = analyze_rnn_output(filepath)

    # ---- SORT VALID PACKETS BY TIME ----
    valid_packets = sort_by_time(valid_packets)

    # ---- METRIC COMPUTATION ----
    protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)
    address_compliance, address_errors = check_address_compliance(valid_packets)
    seq_compliance, seq_errors = check_seq_ordering(valid_packets)
    repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)
    emr, matches = compute_exact_match_rate(Sample_Packets, valid_packets)


    Decodability.append(decodability)
    Protocol_Compliance_Rate.append(protocol_compliance)
    Address_Compliance_Rate.append(address_compliance)
    Seq_Compliance_Rate.append(seq_compliance)
    Repetition_Rate.append(repetition_rate)
    Exact_Match_Rate.append(emr)
    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%, EMR: {emr:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average Exact Match Rate (EMR): {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")


===== SUMMARY REPORT =====

===== RNN PACKET QUALITY REPORT =====
Total lines processed: 402
Valid packets: 382
Corrupt packets: 20
% Corrupt: 4.98%
Timestamp ordering issues: 194

---- CORRUPT PACKETS ----
[Line 0] Error: Empty line
    

[Line 21] Error: Time not numeric
    {"time": "243..058327", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read Attributes Response, Seq: 157"}

[Line 60] Error: Time not numeric
    {"time": "169..930371", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "61", "info": "ZCL knes: Rea Atrriutes Response, Seq: 166"}

[Line 91] Error: Missing required field: protocol
    {"time": "550.748922", "src": "0x1de6", "dst": "0xfffc", "protoocol": "ZigBee", "length": "80", "info": "Link Status"}

[Line 96] Error: Time not numeric
    {"time": "214..903671", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read Attributes Response, Seq: 239"}

[Line 155] Erro

## EXPERIMENT 2

In [42]:
import re

def check_address_compliance_exp2(valid_packets, VALID_SOURCES, VALID_DESTINATIONS):
    """
    Experiment 2 — Address Compliance + Request/Response correctness.
    Differences vs Experiment 1:
        - Response with no request is NOT an error.
        - Request with no response is NOT an error.
        - Only if both exist (same seq), check direction + src!=dst.
    """

    incorrect = []

    # Sort packets by timestamp
    packets_sorted = sorted(valid_packets, key=lambda p: float(p["time"]))

    # Regex for sequence number
    seq_pattern = re.compile(r"Seq:\s*(\d+)")

    # Dictionaries to store requests and responses
    req_dict = {}   # seq → request packet
    res_dict = {}   # seq → response packet

    # ===========================================================
    # PASS 1 — BASIC ADDRESS RULES (apply to EVERY PACKET)
    # ===========================================================
    for i, pkt in enumerate(packets_sorted):

        src = pkt["src"]
        dst = pkt["dst"]
        info = pkt["info"]
        # ----------------------------------------
        # Rule 1 (GAN): Link Status must be broadcast
        # ----------------------------------------
        if "Link Status" in info:
            if dst not in ["0xfffc", "Broadcast"]:
                incorrect.append((i, pkt, 
                    "dst must be 0xfffc or Broadcast for Link Status"))

        # ----------------------------------------
        # Rule 2: src != dst
        # ----------------------------------------
        if src == dst:
            incorrect.append((i, pkt, 
                "src and dst cannot be identical"))

        # --- Rule C: src must be in valid devices ---
        if src not in VALID_SOURCES:
            incorrect.append((i, pkt, f"Invalid src {src}, must be in {VALID_SOURCES}"))

        # --- Rule D: dst must be in valid devices ---
        if dst not in VALID_DESTINATIONS:
            incorrect.append((i, pkt, f"Invalid dst {dst}, must be in {VALID_DESTINATIONS}"))

        # Extract seq number if exists
        match = seq_pattern.search(info)
        if not match:
            continue

        seq = int(match.group(1))

        # Store request or response for later pair-check
        if "ZCL: Read Attributes," in info and "Response" not in info:
            req_dict[seq] = pkt

        elif "ZCL: Read Attributes Response" in info:
            res_dict[seq] = pkt

    # ===========================================================
    # PASS 2 — REQUEST / RESPONSE PAIRS (ONLY if both exist)
    # ===========================================================
    for seq, req_pkt in req_dict.items():

        # If matching response does not exist → ignore (not an error)
        if seq not in res_dict:
            continue

        res_pkt = res_dict[seq]
        req_src = req_pkt["src"]
        req_dst = req_pkt["dst"]

        # --- Rule E1: Response direction must be inverted ---
        if not (res_pkt["src"] == req_dst and res_pkt["dst"] == req_src):
            incorrect.append((
                -1,
                res_pkt,
                f"Seq {seq}: Response direction incorrect. Expected src={req_dst}, dst={req_src}"
            ))

        # --- Rule E2: Response must not have src == dst ---
        if res_pkt["src"] == res_pkt["dst"]:
            incorrect.append((
                -1,
                res_pkt,
                f"Seq {seq}: Response has identical src and dst"
            ))

    # ===========================================================
    # SUMMARY
    # ===========================================================    
    total = len(valid_packets)
    error_count = len(incorrect)
    compliance = 100 * (1 - error_count / total) if total else 0

    print("\n===== ADDRESS COMPLIANCE — EXPERIMENT 2 =====")
    print(f"Total packets: {total}")
    print(f"Violations: {error_count}")
    print(f"Compliance: {compliance:.2f}%")
    print("=============================================\n")

    if incorrect:
        print("---- First Incorrect Packets ----")
        for idx, pkt, reason in incorrect[:20]:
            print(f"[Packet {idx}] Error: {reason}")
            print(pkt)
            print()

    return compliance, incorrect


In [43]:
import re

def check_seq_ordering_exp2(valid_packets, max_step=50):
    """
    Experiment 2 sequence validation:
    
    Rules:
      R1: Seq must exist, must be numeric, and must be in [0,255].
      R2: Request sequences must increase modulo 256 with small steps.
      R3: Response sequences must increase modulo 256 with small steps.
      R4: SAME info type ⇒ seq cannot repeat consecutively.
      R5: No global ordering between request vs response (allowed to interleave).
    """

    sorted_packets = sorted(valid_packets, key=lambda p: float(p["time"]))
    seq_pattern = re.compile(r"Seq:\s*([+-]?\d+)")

    req_seq = []          # numeric sequence list
    res_seq = []          # numeric sequence list
    req_packets = []      # packet objects (parallel array)
    res_packets = []
    req_info = []         # base info prefixes
    res_info = []

    incorrect = []

    # ------------------------------------
    # PASS 1 — Extract & Validate SEQ
    # ------------------------------------
    for idx, pkt in enumerate(sorted_packets):
        info = pkt["info"]

        if "ZCL" not in info:
            continue

        match = seq_pattern.search(info)
        if not match:
            incorrect.append(("MISSING SEQ", pkt, None, None, None,
                              "Missing Seq field in ZCL packet"))
            continue

        raw = match.group(1)

        # Numeric?
        try:
            seq = int(raw)
        except:
            incorrect.append(("NON-NUMERIC SEQ", pkt, None, None, None,
                              f"Non-numeric Seq value: {raw}"))
            continue

        # Range check
        if seq < 0 or seq > 255:
            incorrect.append(("SEQ RANGE ERROR", pkt, None, seq, None,
                              f"Seq out of range: {seq}"))

        # Base info type (everything before Seq:)
        base_info = info.split("Seq:")[0].strip()

        # Classify packet type
        if "ZCL: Read Attributes," in info and "Response" not in info:
            # REQUEST
            req_seq.append(seq)
            req_packets.append(pkt)
            req_info.append(base_info)

        elif "ZCL: Read Attributes Response" in info:
            # RESPONSE
            res_seq.append(seq)
            res_packets.append(pkt)
            res_info.append(base_info)

        # Other ZCL types ignored in Exp2

    # ------------------------------------
    # R2 — Request monotonicity check
    # ------------------------------------
    for i in range(1, len(req_seq)):
        prev = req_seq[i-1]
        curr = req_seq[i]
        diff = (curr - prev) % 256
        pkt = req_packets[i]

        # large backward wrap or same seq
        if diff == 0 or diff > max_step:
            incorrect.append(("REQ ORDER VIOLATION", pkt, prev, curr, diff,
                              "Invalid request sequence jump"))

        # Consecutive repetition for SAME info type
        if req_info[i] == req_info[i-1] and curr == prev:
            incorrect.append(("REQ SAME-TYPE REPEAT", pkt, prev, curr, diff,
                              "Repeated Seq for SAME info type (not allowed)"))

    # ------------------------------------
    # R3 — Response monotonicity check
    # ------------------------------------
    for i in range(1, len(res_seq)):
        prev = res_seq[i-1]
        curr = res_seq[i]
        diff = (curr - prev) % 256
        pkt = res_packets[i]

        if diff == 0 or diff > max_step:
            incorrect.append(("RES ORDER VIOLATION", pkt, prev, curr, diff,
                              "Invalid response sequence jump"))

        if res_info[i] == res_info[i-1] and curr == prev:
            incorrect.append(("RES SAME-TYPE REPEAT", pkt, prev, curr, diff,
                              "Repeated Seq for SAME info type (not allowed)"))

    # ------------------------------------
    # Compute compliance using UNIQUE bad packets
    # ------------------------------------
    # ONLY count packets with errors, not error count
    unique_packets = { id(ev[1]) for ev in incorrect }
    error_packet_count = len(unique_packets)

    total = len(req_seq) + len(res_seq)
    compliance = 100 * (1 - error_packet_count / max(1, total))

    # ------------------------------------
    # REPORT
    # ------------------------------------
    print("\n===== ZCL SEQUENCE ORDER REPORT — EXPERIMENT 2 =====")
    print(f"Total ZCL packets (REQ+RES): {total}")
    print(f"Error events: {len(incorrect)}")
    print(f"Unique packets with errors: {error_packet_count}")
    print(f"Sequence Compliance: {compliance:.2f}%")
    print("====================================================\n")

    for tag, pkt, prev, curr, diff, reason in incorrect[:20]:
        print(f"{tag}: Prev={prev}, Curr={curr}, Jump={diff}")
        print(f"  Time: {pkt['time']}")
        print(f"  Info: {pkt['info']}")
        print(f"  Reason: {reason}\n")

    return compliance, incorrect


In [44]:
REQUIRED_FIELDS = ["time", "src", "dst", "protocol", "length", "info"]
VALID_PROTOCOLS = ["ZigBee", "ZigBee HA"]

VALID_SOURCES = ["0x1de6", "0xd7a7"] # only for experiment #2
VALID_DESTINATIONS = ["0x1de6", "0xd7a7", "0xfffc"]# only for experiment #2

In [45]:
def sort_by_time(packets):
    """Return packets sorted by numeric time value."""
    return sorted(packets, key=lambda p: float(p["time"]))


In [46]:
print("\n===== SUMMARY REPORT =====")

Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []

N = list(range(1,11)) # number of trial

for n in N:

    filepath = f"../Generated_Traffic/TXT_files/RNN_Exp2_Trial_{n}_raw_generated_10_minutes.txt"
    valid_packets, corrupt_packets, decodability = analyze_rnn_output(filepath)

    # ---- SORT VALID PACKETS BY TIME ----
    valid_packets = sort_by_time(valid_packets)

    # ---- METRIC COMPUTATION ----
    protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)
    address_compliance, address_errors = check_address_compliance_exp2(valid_packets, VALID_SOURCES, VALID_DESTINATIONS )
    seq_compliance, seq_errors = check_seq_ordering_exp2(valid_packets)
    repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)
    emr, matches = compute_exact_match_rate(Sample_Packets, valid_packets)


    Decodability.append(decodability)
    Protocol_Compliance_Rate.append(protocol_compliance)
    Address_Compliance_Rate.append(address_compliance)
    Seq_Compliance_Rate.append(seq_compliance)
    Repetition_Rate.append(repetition_rate)
    Exact_Match_Rate.append(emr)

    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average Exact Match Rate (EMR): {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")


===== SUMMARY REPORT =====

===== RNN PACKET QUALITY REPORT =====
Total lines processed: 222
Valid packets: 201
Corrupt packets: 21
% Corrupt: 9.46%
Timestamp ordering issues: 97

---- CORRUPT PACKETS ----
[Line 0] Error: Empty line
    

[Line 4] Error: Time not numeric
    {"time": "14..384536", "src": "0xd7a7", "dst": "0x1de6", "protocol": "ZigBee HA", "length": "68", "info": "ZCL: Read Attributes Response, Seq: 161"}

[Line 18] Error: Time not numeric
    {"time": "259..261645", "src": "0xd7a7", "dst": "0x1de6", "protocol": "ZigBee HA", "length": "52", "info": "ZCL: Read Attributes, Seq: 281"}

[Line 29] Error: Missing required field: dst
    {"time": "235.747494", "src": "0x1de6", "dss": "0xd7a7", "protocol": "ZigBee HA", "length": "56", "info": "ZCL: Read Attributes Response, Seq: 136"}

[Line 30] Error: Missing required field: dst
    {"time": "519.172637", "src": "0xffffc", "protocol": "ZigBee", "length": "50", "info": "Link Status"}

[Line 33] Error: JSON parse error: Expecti

## 30 MINUTES - EXPERIMENT 2 RNN

In [47]:
print("\n===== SUMMARY REPORT =====")

Decodability = []
Protocol_Compliance_Rate =[]
Address_Compliance_Rate =[]
Seq_Compliance_Rate = []
Repetition_Rate =[]
Exact_Match_Rate = []

N = list(range(1,11)) # number of trial

for n in N:

    filepath = f"../Generated_Traffic/TXT_files/RNN_Exp2_Trial_raw_generated_30_minutes.txt"
    valid_packets, corrupt_packets, decodability = analyze_rnn_output(filepath)

    # ---- SORT VALID PACKETS BY TIME ----
    valid_packets = sort_by_time(valid_packets)

    # ---- METRIC COMPUTATION ----
    protocol_compliance, protocol_errors = check_protocol_compliance(valid_packets)
    address_compliance, address_errors = check_address_compliance_exp2(valid_packets, VALID_SOURCES, VALID_DESTINATIONS )
    seq_compliance, seq_errors = check_seq_ordering_exp2(valid_packets)
    repetition_rate, pattern_counts, repeated_packets = check_repetition_rate(valid_packets)
    emr, matches = compute_exact_match_rate(Sample_Packets, valid_packets)


    Decodability.append(decodability)
    Protocol_Compliance_Rate.append(protocol_compliance)
    Address_Compliance_Rate.append(address_compliance)
    Seq_Compliance_Rate.append(seq_compliance)
    Repetition_Rate.append(repetition_rate)
    Exact_Match_Rate.append(emr)

    # print(f"Trial {n}: Decodability: {decodability:.2f}%, Protocol Compliance: {protocol_compliance:.2f}%, Address Compliance: {address_compliance:.2f}%, Seq Compliance: {seq_compliance:.2f}%, Repetition Rate: {repetition_rate:.2f}%")

print("=======================================")
print(f"Average Decodability: {sum(Decodability)/len(Decodability):.2f}%")
print(f"Average Protocol Compliance: {sum(Protocol_Compliance_Rate)/len(Protocol_Compliance_Rate):.2f}%")
print(f"Average Address Compliance: {sum(Address_Compliance_Rate)/len(Address_Compliance_Rate):.2f}%")
print(f"Average Seq Compliance: {sum(Seq_Compliance_Rate)/len(Seq_Compliance_Rate):.2f}%")
print(f"Average Repetition Rate: {sum(Repetition_Rate)/len(Repetition_Rate):.2f}%")
print(f"Average Exact Match Rate (EMR): {sum(Exact_Match_Rate)/len(Exact_Match_Rate):.2f}%")


===== SUMMARY REPORT =====

===== RNN PACKET QUALITY REPORT =====
Total lines processed: 660
Valid packets: 596
Corrupt packets: 64
% Corrupt: 9.70%
Timestamp ordering issues: 292

---- CORRUPT PACKETS ----
[Line 0] Error: Empty line
    

[Line 3] Error: JSON parse error: Expecting ',' delimiter: line 1 column 22 (char 21)
    {"time": "505.4798, "src": "0xd7a7", "dst": "0x1de6", "protocol": "ZigBee HA", "length": "61", "info": "ZCL: Read Attributes Response, Seq: 214"}

[Line 17] Error: Missing required field: info
    {"time": "356.797822", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "56", "ifno": "ZCL: Read Attributes, Seq: 93"}

[Line 19] Error: Time not numeric
    {"time": "390..640873", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "56", "info": "ZCL: Read Attributes Response, Seq: 188"}

[Line 30] Error: Missing required field: dst
    {"time": "24.6334486", "src": "0xd7a7", "ds": "0x1de6", "protocol": "ZigBee HA", "length": "59